# 07 — Regime Feature Contract (provisional)

**Purpose.** Settle, with explicit rules, which inputs the regime model may use and in what form (framework v2.1 Step 4; handoff §25). Each decision below is a **rule evaluated on the data**, so the notebook can be re-run unchanged at 60 days, when the contract becomes binding (board gate n7).

| § | Decision | Rule tested here |
|---|---|---|
| 3 | Asset universe | Minimum data quality per asset (index, perpetual, options) |
| 4 | Pooled vs per-asset | Common volatility factor vs asset-specific persistence |
| 5 | Session boundary | Is there a break in the underlying at 08:00 UTC? |
| 6 | Option representation | Fixed-k vs quote-only fit vs delta: coverage, noise, information |
| 7 | Option-change horizon | Smallest horizon whose change is mostly signal |
| 8 | Redundant pairs | One feature per redundant pair |
| 9 | RV partial windows | Coverage gained vs error from allowing missing bars |
| 10 | Contract v0 | Assembled from the rules above |

**Status: provisional** (~15 sessions). Thresholds are stated up front and not tuned to the outcome. All timestamps UTC; database read-only.

## 1. Environment

In [1]:
from pathlib import Path
import os
import sys
import math
import json
import warnings

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

load_dotenv()
data_root_env = os.getenv("DERIBIT_DATA_ROOT")
if not data_root_env:
    raise RuntimeError("DERIBIT_DATA_ROOT is not defined in the environment/.env file.")
DB_PATH = Path(data_root_env).expanduser().resolve() / "database" / "research.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute("SET TimeZone = 'UTC'")

PROJECT_ROOT = Path(os.getenv("DERIBIT_PROJECT_ROOT", Path.cwd())).resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "data_engineering").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "data_engineering").exists():
    raise RuntimeError("Repository not found: start Jupyter inside the repository or set DERIBIT_PROJECT_ROOT.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from data_engineering.features import regime_inputs as ri
from data_engineering.modeling.clock_baseline import ClockBaselineSet, FeatureSpec

PERIODS_PER_YEAR = ri.PERIODS_PER_YEAR

# Rules, fixed before looking at the outcome.
RULES = {
    "max_zero_return_share": 0.05,        # index must move
    "min_perp_index_corr": 0.80,          # perpetual tracks the index (flow/basis usable)
    "min_fit_acceptance": 0.50,           # quote-only smile present in half the non-settlement states
    "max_node_mark_share": 0.70,          # option surface mostly market quotes
    "max_change_noise_share": 0.30,       # noise share of change variance, -2*rho1
    "pool_min_pc1_share": 0.60,           # common volatility factor share for pooling the scale
    "pool_max_persistence_corr": 0.30,    # above this, persistence states are shared
    "boundary_alpha": 0.05,               # break at 08:00 only if significantly lower dependence
    "rv_partial_max_p90_log_error": 0.10, # partial-window RV error tolerated (log)
}
print(json.dumps(RULES, indent=1))

{
 "max_zero_return_share": 0.05,
 "min_perp_index_corr": 0.8,
 "min_fit_acceptance": 0.5,
 "max_node_mark_share": 0.7,
 "max_change_noise_share": 0.3,
 "pool_min_pc1_share": 0.6,
 "pool_max_persistence_corr": 0.3,
 "boundary_alpha": 0.05,
 "rv_partial_max_p90_log_error": 0.1
}


## 2. Load

In [2]:
cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])


def first_present(*names):
    return next((n for n in names if n in cols), None)


ALIASES = {
    "skew_fixed": first_present("option_skew_025", "skew_025", "local_skew_slope_025"),
    "curv_fixed": first_present("option_curvature_025", "curvature_025"),
}
want = ["state_id", "state_time", "underlying", "session_id", "minutes_to_expiry", "return_5m", "perp_mid_return_5m",
        "futures_index_price", "realized_vol_1h_annualized", "realized_vol_30m_annualized", "index_return_input_eligible",
        "perp_integrity_ok", "vwap", "volume_usd", "futures_tradable_mid_price", "log1p_volume_usd", "trade_count",
        "book_update_count", "is_weekend", "utc_hour", "in_settlement_window",
        "option_iv_atm_forward", "fit_atm_iv", "fit_ok", "fit_skew", "fit_skew_rel", "fit_curvature", "fit_curvature_rel",
        "rr_25d", "bf_25d", "option_atm_quote_ok", "option_shape_quote_ok", "core_smile_usable", "node_mark_share",
        "delta_25d_smile_available",
        "atm_iv_change_15m", "atm_iv_change_30m", "atm_iv_change_1h",
        "skew_025_change_15m", "skew_025_change_30m", "skew_025_change_1h",
        "curvature_025_change_15m", "curvature_025_change_30m", "curvature_025_change_1h",
        "fit_atm_iv_change_15m", "fit_atm_iv_change_30m", "fit_atm_iv_change_1h",
        "fit_skew_change_15m", "fit_skew_change_30m", "fit_skew_change_1h",
        "fit_curvature_change_15m", "fit_curvature_change_30m", "fit_curvature_change_1h"] + [v for v in ALIASES.values() if v]
present = list(dict.fromkeys(c for c in want if c in cols))
print("Not in table (skipped):", [c for c in want if c not in cols])
df = con.execute(f"SELECT {', '.join(present)} FROM time_series_feature_state ORDER BY underlying, state_time").df()
df["state_time"] = pd.to_datetime(df["state_time"], utc=True)


def as_bool(s):
    if s.dtype == object:
        s = s.map({"True": True, "False": False, "true": True, "false": False, True: True, False: False})
    return s.fillna(False).astype(bool)


for c in ["perp_integrity_ok", "fit_ok", "option_atm_quote_ok", "option_shape_quote_ok", "core_smile_usable",
          "in_settlement_window", "is_weekend", "delta_25d_smile_available", "index_return_input_eligible"]:
    if c in df:
        df[c] = as_bool(df[c])
df["r"] = pd.to_numeric(df["return_5m"], errors="coerce")
df["session_date"] = ((df["state_time"] - pd.Timedelta(hours=8)).dt.floor("D") + pd.Timedelta(days=1)).dt.strftime("%Y-%m-%d")

tables = set(con.execute("SELECT table_name FROM duckdb_tables()").df()["table_name"])
if "regime_input_state" in tables:
    inputs = con.execute("SELECT * FROM regime_input_state").df()
else:
    inputs = ri.compute_regime_inputs(df)
df = df.merge(inputs[["state_id"] + [c for c in inputs.columns if c.startswith(("vr_q", "er_n", "ewma_hl", "D_"))]],
              on="state_id", how="left")

dates = sorted(df.loc[df["r"].notna(), "session_date"].unique())
train_dates = set(dates[: len(dates) // 2])
clock = ClockBaselineSet.fit(df[["state_id", "state_time", "underlying", "is_weekend", "return_5m"]],
                             df["session_date"].isin(train_dates),
                             specs=(FeatureSpec("return_5m", "return_5m", transform="log_abs", scale="none"),))
df["f_clock"] = clock.transform(df[["state_id", "state_time", "underlying", "is_weekend", "return_5m"]])[
    "return_5m__clock_vol_factor"].to_numpy()
df["f_clock"] = df["f_clock"].fillna(1.0)
df["r_tilde"] = df["r"] / df["f_clock"]

ASSETS = sorted(df["underlying"].unique())
MAJORS = [a for a in ["BTC", "ETH", "SOL", "XRP"] if a in ASSETS]
ACTIVE = ~df["in_settlement_window"] if "in_settlement_window" in df else pd.Series(True, index=df.index)
print(f"Rows {len(df):,} | assets {ASSETS} | sessions with returns {len(dates)} | clock train {min(train_dates)} → {max(train_dates)}")


def grid_shift(d, column, k):
    out = pd.Series(np.nan, index=d.index)
    for _, g in d.groupby("underlying"):
        g = g.sort_values("state_time")
        v = pd.to_numeric(g[column], errors="coerce").shift(k)
        ok = (g["state_time"] - g["state_time"].shift(k)).abs() == pd.Timedelta(minutes=5 * abs(k))
        out.loc[g.index] = v.where(ok)
    return out

Not in table (skipped): []
Rows 34,272 | assets ['AVAX', 'BTC', 'ETH', 'HYPE', 'SOL', 'TRX', 'XRP'] | sessions with returns 15 | clock train 2026-09-22 → 2026-09-28


## 3. Asset universe

Per asset: share of zero index returns (a frozen or coarse index), correlation of perpetual and index returns (whether perpetual flow and basis describe the index), quote-only fit acceptance and node mark share outside the settlement window (whether the option block is market information), and median book updates (L2 reliability, informational only). An asset enters the **structural** model only if it passes the index and perpetual rules; it gets an **option block** only if it also passes the option rules.

In [3]:
rows = []
for asset, g in df.groupby("underlying"):
    a = g[ACTIVE.loc[g.index]]
    row = {"underlying": asset,
           "zero_return_share": float((g["r"] == 0).sum() / max(g["r"].notna().sum(), 1)),
           "perp_index_corr": float(g[["perp_mid_return_5m", "r"]].corr().iloc[0, 1]) if "perp_mid_return_5m" in g else np.nan,
           "fit_acceptance": float(a["fit_ok"].mean()) if "fit_ok" in a else np.nan,
           "node_mark_share": float(pd.to_numeric(a.get("node_mark_share"), errors="coerce").mean()) if "node_mark_share" in a else np.nan,
           "median_book_updates": float(pd.to_numeric(g.get("book_update_count"), errors="coerce").median()) if "book_update_count" in g else np.nan}
    row["structural_ok"] = (row["zero_return_share"] <= RULES["max_zero_return_share"]
                            and (np.isnan(row["perp_index_corr"]) or row["perp_index_corr"] >= RULES["min_perp_index_corr"]))
    row["option_block_ok"] = (row["structural_ok"] and row["fit_acceptance"] >= RULES["min_fit_acceptance"]
                              and row["node_mark_share"] <= RULES["max_node_mark_share"])
    rows.append(row)
universe = pd.DataFrame(rows).set_index("underlying")
display(universe.round(3))
STRUCTURAL_ASSETS = universe.index[universe["structural_ok"]].tolist()
OPTION_ASSETS = universe.index[universe["option_block_ok"]].tolist()
print("Structural model:", STRUCTURAL_ASSETS)
print("Option block:    ", OPTION_ASSETS)

,zero_return_share,perp_index_corr,fit_acceptance,node_mark_share,median_book_updates,structural_ok,option_block_ok
underlying,,,,,,,
AVAX,0.018,0.973,0.757,0.393,618.0,True,True
BTC,0.000,0.994,0.569,0.688,880.0,True,True
ETH,0.002,0.995,0.639,0.607,883.0,True,True
HYPE,0.002,0.962,0.668,0.566,505.0,True,True
SOL,0.001,0.996,0.614,0.710,858.0,True,False
TRX,0.005,0.214,0.279,0.834,64.0,False,False
XRP,0.020,0.961,0.595,0.589,727.0,True,True


Structural model: ['AVAX', 'BTC', 'ETH', 'HYPE', 'SOL', 'XRP']
Option block:     ['AVAX', 'BTC', 'ETH', 'HYPE', 'XRP']


## 4. Pooled or per-asset?

Two different questions:

- **Volatility (scale).** Non-overlapping hourly clock-adjusted log realized volatility, $\log \sqrt{\sum_{12} \tilde r^2}$, per asset. The share of variance on the first principal component across structural assets measures a common volatility factor. Above the rule threshold, the scale $\sigma_{k,t}$ can share a market component.
- **Persistence (regime character).** Pairwise correlation across assets of the persistence statistics (variance ratio $q=6$, efficiency ratio $N=48$) and of $|D|$ (EWMA, 1 h). If these are weakly correlated, mean-reversion/momentum periods are asset-specific and regimes must be per asset (possibly with a shared volatility component).

Per-asset kurtosis of the standardized returns $z_t = r_t/\sigma^c_{t-1}$ checks whether one emission family can serve all assets.

In [4]:
x = df[df["underlying"].isin(STRUCTURAL_ASSETS)].copy()
x["hour_key"] = x["state_time"].dt.floor("h")
hv = (x.assign(rt2=x["r_tilde"] ** 2).groupby(["underlying", "hour_key"])
      .agg(n=("rt2", "count"), s=("rt2", "sum")))
hv = hv[hv["n"] == 12]
hv["log_rv"] = 0.5 * np.log(hv["s"].where(hv["s"] > 0))
W = hv["log_rv"].unstack("underlying").dropna()
pc1_share = np.nan
if W.shape[1] >= 2 and len(W) > 50:
    Z = (W - W.mean()) / W.std()
    ev = np.linalg.eigvalsh(np.corrcoef(Z.to_numpy().T))[::-1]
    pc1_share = float(ev[0] / ev.sum())
    print(f"Hourly clock-adjusted log RV, {len(W)} complete hours: PC1 share = {pc1_share:.2f}")
    display(W.corr().round(2))

per = {}
for c in [c for c in ["vr_q6_n48", "er_n48"] if c in x]:
    w = x.pivot_table(index="state_time", columns="underlying", values=c).iloc[::12]
    per[c] = w.corr(method="spearman")
x["absD"] = x.get("D_ewma_hl12", pd.Series(np.nan, index=x.index)).abs()
per["abs_D_ewma_hl12"] = x.pivot_table(index="state_time", columns="underlying", values="absD").iloc[::12].corr(method="spearman")
pers_corr = {}
for k, m in per.items():
    vals = m.where(~np.eye(len(m), dtype=bool)).stack()
    pers_corr[k] = float(vals.median()) if len(vals) else np.nan
    print(f"{k}: median pairwise Spearman (non-overlapping hours) = {pers_corr[k]:.2f}")
    display(m.round(2))

kurt = {}
for asset, g in x.groupby("underlying"):
    g = g.sort_values("state_time")
    sig = g["r_tilde"].pow(2).ewm(halflife=12, min_periods=12).mean().pow(0.5) * g["f_clock"]
    z = g["r"] / sig.shift(1)
    kurt[asset] = float(z.replace([np.inf, -np.inf], np.nan).dropna().kurt())
print("Excess kurtosis of standardized returns:", {k: round(v, 1) for k, v in kurt.items()})

POOL_SCALE = bool(pc1_share >= RULES["pool_min_pc1_share"]) if np.isfinite(pc1_share) else False
SHARED_PERSISTENCE = bool(np.nanmedian(list(pers_corr.values())) > RULES["pool_max_persistence_corr"]) if pers_corr else False
print(f"\nCommon volatility component: {POOL_SCALE} | shared persistence states: {SHARED_PERSISTENCE}")

Hourly clock-adjusted log RV, 302 complete hours: PC1 share = 0.63


underlying,AVAX,BTC,ETH,HYPE,SOL,XRP
underlying,,,,,,
AVAX,1.00,0.37,0.48,0.23,0.57,0.60
BTC,0.37,1.00,0.81,0.37,0.68,0.67
ETH,0.48,0.81,1.00,0.33,0.76,0.75
HYPE,0.23,0.37,0.33,1.00,0.30,0.30
SOL,0.57,0.68,0.76,0.30,1.00,0.76
XRP,0.60,0.67,0.75,0.30,0.76,1.00


vr_q6_n48: median pairwise Spearman (non-overlapping hours) = 0.39


underlying,AVAX,BTC,ETH,HYPE,SOL,XRP
underlying,,,,,,
AVAX,1.00,0.28,0.38,0.25,0.41,0.39
BTC,0.28,1.00,0.73,0.18,0.66,0.52
ETH,0.38,0.73,1.00,0.32,0.68,0.61
HYPE,0.25,0.18,0.32,1.00,0.31,0.28
SOL,0.41,0.66,0.68,0.31,1.00,0.66
XRP,0.39,0.52,0.61,0.28,0.66,1.00


er_n48: median pairwise Spearman (non-overlapping hours) = 0.27


underlying,AVAX,BTC,ETH,HYPE,SOL,XRP
underlying,,,,,,
AVAX,1.00,0.14,0.23,0.15,0.29,0.20
BTC,0.14,1.00,0.53,0.19,0.39,0.32
ETH,0.23,0.53,1.00,0.18,0.41,0.39
HYPE,0.15,0.19,0.18,1.00,0.27,0.14
SOL,0.29,0.39,0.41,0.27,1.00,0.48
XRP,0.20,0.32,0.39,0.14,0.48,1.00


abs_D_ewma_hl12: median pairwise Spearman (non-overlapping hours) = 0.27


underlying,AVAX,BTC,ETH,HYPE,SOL,XRP
underlying,,,,,,
AVAX,1.00,0.20,0.23,0.23,0.27,0.26
BTC,0.20,1.00,0.59,0.20,0.44,0.35
ETH,0.23,0.59,1.00,0.29,0.51,0.37
HYPE,0.23,0.20,0.29,1.00,0.20,0.23
SOL,0.27,0.44,0.51,0.20,1.00,0.44
XRP,0.26,0.35,0.37,0.23,0.44,1.00


Excess kurtosis of standardized returns: {'AVAX': 5.9, 'BTC': 3.2, 'ETH': 7.6, 'HYPE': 4.3, 'SOL': 6.5, 'XRP': 4.4}

Common volatility component: True | shared persistence states: False


## 5. Session boundary

The index is continuous through 08:00 UTC; only the option universe changes. If nothing in the underlying breaks at 08:00, the structural chain should **continue** across sessions (no forced restart, no initial-state reset), while session objects (session VWAP, option block) reset.

Test: lag-1 correlation of $\log|\tilde r_t|$ for pairs that straddle 08:00 (07:55→08:00 and 08:00→08:05) against all other adjacent pairs, compared with a Fisher $z$ test,

$$
Z = \frac{\operatorname{atanh}\hat\rho_{\text{other}} - \operatorname{atanh}\hat\rho_{08}}{\sqrt{1/(n_{08}-3) + 1/(n_{\text{other}}-3)}},
$$

per asset and pooled over the structural assets (about two pairs per session and asset, so single-asset tests have little power). A break is declared only if the pooled one-sided test rejects. The mean of $|\tilde r|$ in the hour around 08:00 relative to the asset mean is shown as a descriptive check.

In [5]:
from math import atanh, erf, sqrt


def fisher_test(r_b, n_b, r_o, n_o):
    if n_b < 5 or not np.isfinite(r_b):
        return np.nan, np.nan
    z = (atanh(r_o) - atanh(r_b)) / sqrt(1 / (n_b - 3) + 1 / (n_o - 3))
    return z, 0.5 * (1 - erf(z / sqrt(2)))   # one-sided: dependence lower across 08:00


b_rows, pooled_pairs = [], {"b": [], "o": []}
for asset, g in df[df["underlying"].isin(STRUCTURAL_ASSETS)].groupby("underlying"):
    g = g.sort_values("state_time")
    la = np.log(g["r_tilde"].abs().where(g["r_tilde"].abs() > 0))
    prev = la.shift(1)
    ok = (g["state_time"].diff() == pd.Timedelta(minutes=5)) & la.notna() & prev.notna()
    mod = g["state_time"].dt.hour * 60 + g["state_time"].dt.minute
    straddle = mod.isin([480, 485])
    acf_b = np.corrcoef(la[ok & straddle], prev[ok & straddle])[0, 1] if (ok & straddle).sum() > 10 else np.nan
    acf_o = np.corrcoef(la[ok & ~straddle], prev[ok & ~straddle])[0, 1]
    near = mod.between(450, 510)
    n_b, n_o = int((ok & straddle).sum()), int((ok & ~straddle).sum())
    zst, p = fisher_test(acf_b, n_b, acf_o, n_o)
    # demean per asset before pooling so level differences do not create correlation
    for key, m in [("b", ok & straddle), ("o", ok & ~straddle)]:
        pooled_pairs[key].append(pd.DataFrame({"x": la[m] - la[ok].mean(), "y": prev[m] - prev[ok].mean()}))
    b_rows.append({"underlying": asset, "pairs_at_08": n_b, "acf1_across_08": acf_b, "acf1_elsewhere": acf_o,
                   "fisher_z": zst, "p_lower_at_08": p,
                   "mean_abs_rtilde_07:30-08:30": float(g.loc[near, "r_tilde"].abs().mean() / g["r_tilde"].abs().mean())})
boundary = pd.DataFrame(b_rows).set_index("underlying")
display(boundary.round(3))
pb, po = pd.concat(pooled_pairs["b"]), pd.concat(pooled_pairs["o"])
rb_, ro_ = np.corrcoef(pb["x"], pb["y"])[0, 1], np.corrcoef(po["x"], po["y"])[0, 1]
z_pool, p_pool = fisher_test(rb_, len(pb), ro_, len(po))
CONTINUOUS_ACROSS_SESSIONS = not (np.isfinite(p_pool) and p_pool < RULES["boundary_alpha"])
print(f"Pooled: lag-1 correlation across 08:00 {rb_:.3f} (n={len(pb)}) vs elsewhere {ro_:.3f} (n={len(po)}); "
      f"Fisher z {z_pool:.2f}, one-sided p {p_pool:.3f}")
print(f"Underlying chain continuous across 08:00: {CONTINUOUS_ACROSS_SESSIONS}")

,pairs_at_08,acf1_across_08,acf1_elsewhere,fisher_z,p_lower_at_08,mean_abs_rtilde_07:30-08:30
underlying,,,,,,
AVAX,28,0.278,0.158,-0.629,0.735,1.024
BTC,28,-0.121,0.089,1.050,0.147,0.951
ETH,28,0.357,0.069,-1.516,0.935,0.991
HYPE,28,-0.026,0.065,0.455,0.324,0.952
SOL,28,0.189,0.091,-0.499,0.691,0.994
XRP,27,0.384,0.157,-1.206,0.886,0.922


Pooled: lag-1 correlation across 08:00 0.197 (n=167) vs elsewhere 0.101 (n=24107); Fisher z -1.26, one-sided p 0.896
Underlying chain continuous across 08:00: True


## 6. Option representation

Three representations of the same smile:

| Family | ATM level | Skew | Curvature | Missing when |
|---|---|---|---|---|
| fixed-k | `option_iv_atm_forward` | fixed-k 25 skew | fixed-k 25 curvature | rarely (mark-filled) |
| quote-only fit | `fit_atm_iv` | `fit_skew_rel` | `fit_curvature_rel` | fewer than 5 quotes spanning both wings |
| delta | — | `rr_25d` | `bf_25d` | 25-delta nodes not interpolable |

Measured per asset, outside the settlement window: **coverage** (all states, and the quote-backed share for fixed-k), **persistence** of the level (lag-6 autocorrelation, 30 minutes), and **information**: Spearman correlation of the ATM level with the realized volatility of the next hour (index), next to the trailing 1-hour RV as a benchmark. A representation that is available but mark-filled carries the exchange's model, not market information.

In [6]:
d = df.sort_values(["underlying", "state_time"]).copy()
r2 = d["r"] ** 2
d["_r2"] = r2
fwd = sum(grid_shift(d, "_r2", -k) for k in range(1, 13))
d["log_fwd_rv1h"] = 0.5 * np.log((fwd / 12 * PERIODS_PER_YEAR).where(fwd > 0))
d["log_rv1h"] = np.log(pd.to_numeric(d["realized_vol_1h_annualized"], errors="coerce").where(lambda s: s > 0))

FAMILIES = {
    "fixed-k": {"atm": "option_iv_atm_forward", "skew": ALIASES["skew_fixed"], "curv": ALIASES["curv_fixed"]},
    "fit": {"atm": "fit_atm_iv", "skew": "fit_skew_rel", "curv": "fit_curvature_rel"},
    "delta": {"atm": None, "skew": "rr_25d", "curv": "bf_25d"},
}
rows = []
for asset, g in d[ACTIVE.loc[d.index]].groupby("underlying"):
    for fam, m in FAMILIES.items():
        for role, c in m.items():
            if c is None or c not in g:
                continue
            v = pd.to_numeric(g[c], errors="coerce")
            if fam == "fit":
                v = v.where(g["fit_ok"]) if "fit_ok" in g else v
            row = {"underlying": asset, "family": fam, "role": role, "column": c, "coverage": float(v.notna().mean())}
            if fam == "fixed-k" and role == "atm" and "option_atm_quote_ok" in g:
                row["quote_backed_coverage"] = float((v.notna() & g["option_atm_quote_ok"]).mean())
            lag6 = grid_shift(g.assign(_v=v), "_v", 6)
            row["acf_lag6"] = float(pd.concat([v, lag6], axis=1).dropna().corr().iloc[0, 1]) if v.notna().sum() > 100 else np.nan
            if role == "atm":
                lv = np.log(v.where(v > 0))
                pair = pd.concat([lv, g["log_fwd_rv1h"]], axis=1).dropna()
                row["spearman_fwd_rv1h"] = float(pair.corr(method="spearman").iloc[0, 1]) if len(pair) > 100 else np.nan
                bench = g[["log_rv1h", "log_fwd_rv1h"]].dropna()
                row["benchmark_trailing_rv"] = float(bench.corr(method="spearman").iloc[0, 1]) if len(bench) > 100 else np.nan
            rows.append(row)
representation = pd.DataFrame(rows)
display(representation.pivot_table(index=["role", "family"], columns="underlying", values="coverage").round(2))
atm = representation[representation.role == "atm"]
display(atm.pivot_table(index="family", columns="underlying",
                        values=["spearman_fwd_rv1h", "benchmark_trailing_rv", "quote_backed_coverage"]).round(2))
display(representation.pivot_table(index=["role", "family"], columns="underlying", values="acf_lag6").round(2))

underlying    AVAX   BTC   ETH  HYPE   SOL   TRX   XRP
role family                                           
atm  fit      0.76  0.57  0.64  0.67  0.61  0.28  0.60
     fixed-k  0.92  0.92  0.92  0.92  0.92  0.92  0.92
curv delta    0.85  0.60  0.64  0.75  0.81  0.37  0.73
     fit      0.76  0.57  0.64  0.67  0.61  0.28  0.60
     fixed-k  0.92  0.92  0.92  0.92  0.92  0.92  0.90
skew delta    0.85  0.60  0.64  0.75  0.81  0.37  0.73
     fit      0.76  0.57  0.64  0.67  0.61  0.28  0.60
     fixed-k  0.92  0.92  0.92  0.92  0.92  0.92  0.90

benchmark_trailing_rv                                    quote_backed_coverage                                   spearman_fwd_rv1h                                    
underlying                  AVAX   BTC   ETH  HYPE   SOL  TRX   XRP                  AVAX   BTC   ETH  HYPE   SOL  TRX  XRP              AVAX   BTC   ETH  HYPE   SOL   TRX   XRP
family                                                                                                                                                                           
fit                         0.58  0.67  0.65  0.58  0.59  0.4  0.71                   NaN   NaN   NaN   NaN   NaN  NaN  NaN              0.42  0.48  0.63  0.49  0.51  0.37  0.61
fixed-k                     0.58  0.67  0.65  0.58  0.59  0.4  0.71                  0.86  0.86  0.87  0.86  0.82  0.7  0.8              0.44  0.65  0.66  0.50  0.58  0.33  0.69

underlying    AVAX   BTC   ETH  HYPE   SOL   TRX   XRP
role family                                           
atm  fit      0.99  0.97  0.98  0.97  0.96  0.91  0.98
     fixed-k  0.99  0.76  0.87  0.95  0.94  0.93  0.98
curv delta    0.49  0.12  0.12  0.53  0.42  0.42  0.61
     fit      0.56  0.74  0.82  0.60  0.69  0.71  0.81
     fixed-k  0.82  0.67  0.84  0.73  0.82  0.97  0.85
skew delta    0.62  0.30  0.77  0.84  0.69  0.52  0.83
     fit      0.62  0.93  0.95  0.83  0.76  0.44  0.84
     fixed-k  0.72  0.91  0.91  0.65  0.80  0.95  0.81

## 7. Option-change horizon

For a level $s_t$ observed with i.i.d. measurement noise of variance $\sigma_n^2$, non-overlapping changes over $k$ bars have lag-1 autocorrelation

$$
\rho_1(k) = \frac{-\sigma_n^2}{\sigma^2_{\Delta s}(k) + 2\sigma_n^2}
\quad\Rightarrow\quad
\text{noise share of } \operatorname{Var}(\Delta_k) = \frac{2\sigma_n^2}{\sigma^2_{\Delta s}(k) + 2\sigma_n^2} = -2\rho_1(k).
$$

The signal variance grows with $k$ while the noise variance does not, so the noise share falls with the horizon. The contract uses the **smallest horizon whose median noise share across option assets is below the rule threshold**. Changes are source-consistent (Notebook 03); outside the settlement window.

In [7]:
CHANGES = {
    ("fixed-k", "atm"): {"15m": "atm_iv_change_15m", "30m": "atm_iv_change_30m", "1h": "atm_iv_change_1h"},
    ("fixed-k", "skew"): {"15m": "skew_025_change_15m", "30m": "skew_025_change_30m", "1h": "skew_025_change_1h"},
    ("fixed-k", "curv"): {"15m": "curvature_025_change_15m", "30m": "curvature_025_change_30m", "1h": "curvature_025_change_1h"},
    ("fit", "atm"): {"15m": "fit_atm_iv_change_15m", "30m": "fit_atm_iv_change_30m", "1h": "fit_atm_iv_change_1h"},
    ("fit", "skew"): {"15m": "fit_skew_change_15m", "30m": "fit_skew_change_30m", "1h": "fit_skew_change_1h"},
    ("fit", "curv"): {"15m": "fit_curvature_change_15m", "30m": "fit_curvature_change_30m", "1h": "fit_curvature_change_1h"},
}
BARS = {"15m": 3, "30m": 6, "1h": 12}
rows = []
dd = d[ACTIVE.loc[d.index]]
for (fam, role), hz in CHANGES.items():
    for h, c in hz.items():
        if c not in dd:
            continue
        lagged = grid_shift(d, c, BARS[h])
        for asset, g in dd.groupby("underlying"):
            if asset not in OPTION_ASSETS and OPTION_ASSETS:
                continue
            pair = pd.concat([pd.to_numeric(g[c], errors="coerce"), lagged.loc[g.index]], axis=1).dropna()
            if len(pair) < 100:
                continue
            rho = float(pair.corr().iloc[0, 1])
            rows.append({"family": fam, "role": role, "horizon": h, "underlying": asset, "n": len(pair),
                         "rho1": rho, "noise_share": float(np.clip(-2 * rho, 0, 1))})
horizon = pd.DataFrame(rows)
if len(horizon):
    hz_tab = horizon.pivot_table(index=["family", "role"], columns="horizon", values="noise_share", aggfunc="median")
    hz_tab = hz_tab[[h for h in ["15m", "30m", "1h"] if h in hz_tab.columns]]
    display(hz_tab.round(2))
    CHOSEN_HORIZON = {}
    for key, row in hz_tab.iterrows():
        ok = [h for h in hz_tab.columns if np.isfinite(row[h]) and row[h] <= RULES["max_change_noise_share"]]
        CHOSEN_HORIZON["/".join(key)] = ok[0] if ok else None
    print("Smallest admissible change horizon:", CHOSEN_HORIZON)
else:
    CHOSEN_HORIZON = {}

horizon        15m   30m    1h
family  role                  
fit     atm   0.11  0.00  0.00
        curv  0.74  0.66  0.62
        skew  0.51  0.54  0.45
fixed-k atm   0.15  0.06  0.07
        curv  0.52  0.45  0.34
        skew  0.46  0.36  0.24

Smallest admissible change horizon: {'fit/atm': '15m', 'fit/curv': None, 'fit/skew': None, 'fixed-k/atm': '15m', 'fixed-k/curv': None, 'fixed-k/skew': '1h'}


## 8. Redundant pairs

Notebook 05 found these pairs nearly interchangeable (Spearman ≥ 0.8). For each, the contract keeps the member with **market information and coverage**: quote-backed before mark-filled; then the higher coverage in the active session; then the lower noise share of its 30-minute change. The correlation is recomputed here.

In [8]:
PAIRS = [("option_iv_atm_forward", "fit_atm_iv"), ("fit_skew_rel", "rr_25d"),
         (ALIASES["skew_fixed"], "fit_skew_rel"), ("log1p_volume_usd", "trade_count")]
noise30 = {}
if len(horizon):
    for (fam, role), hz in CHANGES.items():
        s = horizon[(horizon.family == fam) & (horizon.role == role) & (horizon.horizon == "30m")]["noise_share"]
        if len(s):
            noise30[{"fixed-k": {"atm": "option_iv_atm_forward", "skew": ALIASES["skew_fixed"], "curv": ALIASES["curv_fixed"]},
                     "fit": {"atm": "fit_atm_iv", "skew": "fit_skew_rel", "curv": "fit_curvature_rel"}}[fam][role]] = float(s.median())
QUOTE_BACKED = {"fit_atm_iv", "fit_skew_rel", "fit_curvature_rel", "log1p_volume_usd", "trade_count"}
rows = []
act = d[ACTIVE.loc[d.index] & d["underlying"].isin(STRUCTURAL_ASSETS)]
for a_, b_ in PAIRS:
    if a_ is None or a_ not in act or b_ not in act:
        continue
    all_rows = pd.Series(True, index=act.index)
    va = pd.to_numeric(act[a_], errors="coerce").where(act["fit_ok"] if a_.startswith("fit_") else all_rows)
    vb = pd.to_numeric(act[b_], errors="coerce").where(act["fit_ok"] if b_.startswith("fit_") else all_rows)
    rho = float(pd.concat([va, vb], axis=1).dropna().corr(method="spearman").iloc[0, 1])

    def score(c, v):
        return (c in QUOTE_BACKED, round(float(v.notna().mean()), 2), -noise30.get(c, 0.5))

    keep = max([(a_, va), (b_, vb)], key=lambda cv: score(*cv))[0]
    rows.append({"pair": f"{a_} ~ {b_}", "spearman": rho, f"coverage_{a_}": float(va.notna().mean()),
                 f"coverage_{b_}": float(vb.notna().mean()), "keep": keep})
redundant = pd.DataFrame(rows)
display(redundant.round(3))

,pair,spearman,coverage_option_iv_atm_forward,coverage_fit_atm_iv,keep,coverage_fit_skew_rel,coverage_rr_25d,coverage_option_skew_025,coverage_log1p_volume_usd,coverage_trade_count
0,option_iv_atm_forward ~ fit_atm_iv,0.999,0.923,0.64,fit_atm_iv,NaN,NaN,NaN,NaN,NaN
1,fit_skew_rel ~ rr_25d,0.898,NaN,NaN,fit_skew_rel,0.64,0.729,NaN,NaN,NaN
2,option_skew_025 ~ fit_skew_rel,0.883,NaN,NaN,fit_skew_rel,0.64,NaN,0.92,NaN,NaN
3,log1p_volume_usd ~ trade_count,0.791,NaN,NaN,log1p_volume_usd,NaN,NaN,NaN,0.875,0.875


## 9. Realized volatility with missing bars

A complete-window rule loses the whole 1-hour RV when one of its 12 returns is missing. A partial rule rescales the available bars,

$$
\widehat{\mathrm{RV}}_{1h} = \sqrt{\frac{12}{m}\sum_{i \in \text{available}} r_i^2 \cdot \frac{\mathrm{PERIODS\_PER\_YEAR}}{12}}, \qquad m \ge m_{\min},
$$

The error is measured on complete windows by deleting $12-m$ random bars and comparing with the complete RV ($|\log|$ ratio). The table shows the coverage gained and the 90th percentile error for $m_{\min} = 11, 10, 9$. The contract allows the smallest $m_{\min}$ whose p90 error stays within the rule.

In [9]:
rng = np.random.default_rng(7)
rows = []
for asset, g in df[df["underlying"].isin(STRUCTURAL_ASSETS)].groupby("underlying"):
    g = g.sort_values("state_time")
    grid = pd.date_range(g["state_time"].min(), g["state_time"].max(), freq="5min")
    r = g.set_index("state_time")["r"].reindex(grid).to_numpy()
    windows = np.lib.stride_tricks.sliding_window_view(r, 12)
    n_ok = np.isfinite(windows).sum(axis=1)
    complete = windows[n_ok == 12]
    base = np.log(np.sqrt((complete ** 2).sum(axis=1)))
    for m in [11, 10, 9]:
        errs = []
        for w, b0 in zip(complete[::3], base[::3]):
            keep = rng.choice(12, size=m, replace=False)
            errs.append(abs(np.log(np.sqrt((w[keep] ** 2).sum() * 12 / m)) - b0))
        rows.append({"underlying": asset, "m_min": m, "complete_share": float((n_ok == 12).mean()),
                     "partial_share": float((n_ok >= m).mean()), "p50_log_error": float(np.median(errs)),
                     "p90_log_error": float(np.quantile(errs, 0.9))})
rv_partial = pd.DataFrame(rows)
display(rv_partial.pivot_table(index="m_min", columns="underlying", values=["partial_share", "p90_log_error"]).round(3))
tab = rv_partial.groupby("m_min")["p90_log_error"].median()
ok = [m for m in sorted(tab.index) if tab[m] <= RULES["rv_partial_max_p90_log_error"]]
RV_MIN_BARS = min(ok) if ok else 12
print(f"Median complete-window share {rv_partial['complete_share'].median():.3f}; allowed minimum bars per 1h RV: {RV_MIN_BARS}")

p90_log_error                                    partial_share                                   
underlying          AVAX    BTC    ETH   HYPE    SOL    XRP          AVAX    BTC    ETH   HYPE    SOL    XRP
m_min                                                                                                       
9                  0.228  0.239  0.217  0.214  0.216  0.219         0.855  0.855  0.855  0.855  0.855  0.855
10                 0.161  0.154  0.161  0.158  0.164  0.157         0.847  0.847  0.847  0.847  0.847  0.847
11                 0.090  0.085  0.101  0.092  0.093  0.090         0.761  0.761  0.761  0.761  0.761  0.761

Median complete-window share 0.745; allowed minimum bars per 1h RV: 11


## 10. Contract v0 (provisional)

Assembled from the rules above plus decisions already recorded (Decisions §1–7):

- structural returns, RV and displacement from the Deribit index; perpetual only through gated flow/basis;
- every structural and option input passes the training-only clock baseline before it enters an emission;
- the option block is a **masked** emission: absent late in the session and at weekends by construction (Notebook 06), never imputed;
- 5-minute option changes are excluded (Notebook 03).

The JSON below is the machine-readable form. It becomes binding only after the 60-day re-run; changes between now and then must be explained by the rules, not by outcomes.

In [10]:
def keep_of(column_a, column_b, default):
    if len(redundant):
        hit = redundant[redundant["pair"] == f"{column_a} ~ {column_b}"]
        if len(hit):
            return hit["keep"].iloc[0]
    return default


atm_choice = keep_of("option_iv_atm_forward", "fit_atm_iv", "fit_atm_iv")
skew_choice = keep_of("fit_skew_rel", "rr_25d", "fit_skew_rel")
contract = {
    "status": "provisional",
    "sessions": len(dates),
    "rules": RULES,
    "assets": {"structural": STRUCTURAL_ASSETS, "option_block": OPTION_ASSETS},
    "pooling": {"common_volatility_component": POOL_SCALE, "pc1_share": pc1_share,
                "shared_persistence_states": SHARED_PERSISTENCE, "persistence_cross_corr": pers_corr,
                "regimes": "per asset" if not SHARED_PERSISTENCE else "pooled candidates"},
    "session_boundary": {"underlying_chain": "continuous" if CONTINUOUS_ACROSS_SESSIONS else "restart at 08:00",
                         "reset_at_08": ["session VWAP", "option block"]},
    "structural_inputs": {
        "returns": "index return_5m / clock factor f_t",
        "scale": "sigma^c = f_t * EWMA(r~^2), half-life 12",
        "displacement": "D^c from regime_input_state gaps (anchor choice pending Notebook 06b)",
        "persistence": ["vr_q6_n48", "er_n48"],
        "rv_min_bars_per_hour": RV_MIN_BARS,
    },
    "option_block": {
        "mask": "fit_ok and not in_settlement_window",
        "atm_level": atm_choice,
        "skew": skew_choice,
        "curvature": "fit_curvature_rel",
        "change_horizon": CHOSEN_HORIZON,
        "excluded": ["all 5m option changes", "mark-filled nodes as signals"],
    },
    "activity": keep_of("log1p_volume_usd", "trade_count", "log1p_volume_usd") + " (description only; not in the scale, Notebook 05)",
}
print(json.dumps(contract, indent=1, default=lambda o: float(o) if isinstance(o, (np.floating,)) else str(o)))

{
 "status": "provisional",
 "sessions": 15,
 "rules": {
  "max_zero_return_share": 0.05,
  "min_perp_index_corr": 0.8,
  "min_fit_acceptance": 0.5,
  "max_node_mark_share": 0.7,
  "max_change_noise_share": 0.3,
  "pool_min_pc1_share": 0.6,
  "pool_max_persistence_corr": 0.3,
  "boundary_alpha": 0.05,
  "rv_partial_max_p90_log_error": 0.1
 },
 "assets": {
  "structural": [
   "AVAX",
   "BTC",
   "ETH",
   "HYPE",
   "SOL",
   "XRP"
  ],
  "option_block": [
   "AVAX",
   "BTC",
   "ETH",
   "HYPE",
   "XRP"
  ]
 },
 "pooling": {
  "common_volatility_component": true,
  "pc1_share": 0.629340654155989,
  "shared_persistence_states": false,
  "persistence_cross_corr": {
   "vr_q6_n48": 0.39420589388162347,
   "er_n48": 0.26580648087258213,
   "abs_D_ewma_hl12": 0.2676809721614147
  },
  "regimes": "per asset"
 },
 "session_boundary": {
  "underlying_chain": "continuous",
  "reset_at_08": [
   "session VWAP",
   "option block"
  ]
 },
 "structural_inputs": {
  "returns": "index return_5m /

In [11]:
con.close()
print("DuckDB connection closed.")

DuckDB connection closed.
